In [1]:
%reset -f

In [2]:
!sudo apt update > /dev/null; sudo apt install -y gcc > /dev/null

In [3]:
!pip install torch > /dev/null

In [4]:
!pip install torch_geometric > /dev/null

In [5]:
!pip install pytorch_lightning > /dev/null

In [6]:
!pip install torchmetrics > /dev/null

In [7]:
!pip install -r ./../r.txt > /dev/null

ERROR: Could not open requirements file: [Errno 2] No such file or directory: './../r.txt'


In [101]:
import time

running_time = time.time()

## implementing the exemple in (https://graphein.ai/notebooks/pscdb_baselines.html)

In [1]:
import sys
sys.path.append('../')

import logging
logging.getLogger('matplotlib').setLevel(logging.CRITICAL)
logging.getLogger('graphein').setLevel(logging.INFO)

In [2]:
import pandas as pd
import numpy as np

import torch
import torch.nn as nn
import pytorch_lightning as pl
from tqdm import tqdm
import networkx as nx
import torch_geometric
from torch_geometric.data import Data
from torch_geometric.utils import from_networkx
from sklearn.preprocessing import LabelBinarizer
from sklearn.metrics import f1_score

import warnings
warnings.filterwarnings("ignore")


In [3]:
df = pd.read_csv("../../data/structural_rearrangement_data.csv")
pdbs = df["Free PDB"]
y = [torch.argmax(torch.Tensor(lab)).type(torch.LongTensor) for lab in LabelBinarizer().fit_transform(df.motion_type)]

In [4]:
import pickle as pk

In [5]:
from graphein.protein.config import ProteinGraphConfig
from graphein.protein.edges.distance import (
    add_hydrogen_bond_interactions, 
    add_peptide_bonds, 
    add_k_nn_edges, 
    add_aromatic_interactions,
    add_aromatic_sulphur_interactions,
)
from graphein.protein.graphs import construct_graph
from graphein.protein.utils import download_pdb
import os

from functools import partial

In [6]:
def fit(graph: nx.Graph) -> nx.Graph:
    g_config = graph.graph["config"]
    g_pdb_code = graph.graph["pdb_code"]
    graph.graph.clear()
    graph.graph['config'] = g_config
    graph.graph['pdb_code'] = g_pdb_code

    return graph

In [7]:
# Override config with constructors
constructors = {
    # "edge_construction_functions": [partial(add_k_nn_edges, k=3, long_interaction_threshold=0)],
    "graph_metadata_functions": [fit],
    "edge_construction_functions": [add_hydrogen_bond_interactions, add_aromatic_sulphur_interactions, add_aromatic_interactions, add_peptide_bonds, partial(add_k_nn_edges, k=3, long_interaction_threshold=0)],
    'granularity': 'CA',
    #"node_metadata_functions": [add_dssp_feature]
}

config = ProteinGraphConfig(**constructors)
print(config.dict())

{'granularity': 'CA', 'keep_hets': [], 'insertions': True, 'alt_locs': 'max_occupancy', 'pdb_dir': None, 'verbose': False, 'exclude_waters': True, 'deprotonate': False, 'protein_df_processing_functions': None, 'edge_construction_functions': [<function add_hydrogen_bond_interactions at 0x7ff5e5ac80e0>, <function add_aromatic_sulphur_interactions at 0x7ff5e5ac82c0>, <function add_aromatic_interactions at 0x7ff5e5ac8220>, <function add_peptide_bonds at 0x7ff5e5ab3ec0>, functools.partial(<function add_k_nn_edges at 0x7ff5e5ac8ae0>, k=3, long_interaction_threshold=0)], 'node_metadata_functions': [<function meiler_embedding at 0x7ff5e5acade0>], 'edge_metadata_functions': None, 'graph_metadata_functions': [<function fit at 0x7ff5e50d3380>], 'get_contacts_config': None, 'dssp_config': None}


In [8]:
graph_list = []
y_list = []

pdb_data_path = "../../data/pdb_files/"

for idx, pdb in enumerate(tqdm(pdbs)):
    if os.path.exists(f'{pdb_data_path}/{pdb}.pdb'):
        pdb_file = os.path.abspath(f'{pdb_data_path}/{pdb}.pdb')
    else:
        try:
            pdb_file = download_pdb(pdb, f'{pdb_data_path}/')
        except Exception as e:
            print(str(idx) + "processing error....")
            pass
    
    if pdb_file == None:
        print(str(idx) + "processing error....")
        pass
    
    try:
        graph = construct_graph(config=config, path=pdb_file, pdb_code=pdb)
            
        graph_list.append(graph)
        y_list.append(y[idx])
    except:
        print(str(idx) + ' processing error...')
        pass

graph = None

  0%|          | 0/891 [00:00<?, ?it/s]

Output()

  0%|          | 1/891 [00:00<03:13,  4.60it/s]

Output()

  0%|          | 2/891 [00:00<02:40,  5.55it/s]

...

Output()

100%|█████████▉| 889/891 [03:04<00:00,  6.80it/s]

Output()

Output()

100%|██████████| 891/891 [03:04<00:00,  4.82it/s]


In [10]:
from pympler.asizeof import asizeof

print(asizeof(graph_list)/1024/1024)

766.7004318237305


In [112]:
from graphein.ml.conversion import GraphFormatConvertor

format_convertor = GraphFormatConvertor('nx', 'pyg',
                                        verbose = 'gnn',
                                        columns = None)

In [113]:
pyg_list = [format_convertor(graph) for graph in tqdm(graph_list)]

100%|██████████| 891/891 [00:08<00:00, 104.90it/s]


In [11]:
print(len(pdbs))

891


In [ ]:
print(len(graph_))

In [114]:
for idx, g in enumerate(pyg_list):
    g.y = y_list[idx]
    g.coords = torch.FloatTensor(g.coords)

In [115]:
pyg_list = [
    g for g in pyg_list
    if g.coords.shape[0] == len(g.node_id)
]

In [116]:
config_default = dict(
    n_hid = 8,
    n_out = 8,
    batch_size = 4,
    dropout = 0.5,
    lr = 0.001,
    num_heads = 32,
    num_att_dim = 64,
    model_name = 'GCN'
)

class Struct:
    def __init__(self, **entries):
        self.__dict__.update(entries)

config = Struct(**config_default)

global model_name
model_name = config.model_name

In [117]:
import numpy as np
np.random.seed(42)
idx_all = np.arange(len(pyg_list))
np.random.shuffle(idx_all)

train_idx, valid_idx, test_idx = np.split(idx_all, [int(.8*len(pyg_list)), int(.9*len(pyg_list))])
train, valid, test = [pyg_list[i] for i in train_idx], [pyg_list[i] for i in valid_idx], [pyg_list[i] for i in test_idx]

from torch_geometric.data import DataLoader
train_loader = DataLoader(train, batch_size=config.batch_size, shuffle = True, drop_last = True)
valid_loader = DataLoader(valid, batch_size=32)
test_loader = DataLoader(test, batch_size=32)

In [118]:
pyg_list[0]

Data(edge_index=[2, 2274], node_id=[635], coords=[635, 3], num_nodes=635, y=1)

In [119]:
from torch_geometric.nn import GCNConv, GATConv, SAGEConv, global_add_pool
from torch.nn.functional import mse_loss, nll_loss, relu, softmax, cross_entropy
from torch.nn import functional as F
from torchmetrics.functional import accuracy

In [120]:
class GraphNets(pl.LightningModule):
    def __init__(self):
        super().__init__()

        if model_name == 'GCN':
            self.layer1 = GCNConv(in_channels=3, out_channels=config.n_hid)
            self.layer2 = GCNConv(in_channels=config.n_hid, out_channels=config.n_out)

        elif model_name == 'GAT':
            self.layer1 = GATConv(3, config.num_att_dim, heads=config.num_heads, dropout=config.dropout)
            self.layer2 = GATConv(config.num_att_dim * config.num_heads, out_channels = config.n_out, heads=1, concat=False,
                                 dropout=config.dropout)

        elif model_name == 'GraphSAGE':
            self.layer1 = SAGEConv(3, config.n_hid)
            self.layer2 = SAGEConv(config.n_hid, config.n_out)

        self.decoder = nn.Linear(config.n_out, 7)

    def forward(self, g):
        x = g.coords
        x = F.dropout(x, p=config.dropout, training=self.training)
        x = F.elu(self.layer1(x, g.edge_index))
        x = F.dropout(x, p=config.dropout, training=self.training)
        x = self.layer2(x, g.edge_index)
        x = global_add_pool(x, batch=g.batch)
        x = self.decoder(x)
        return softmax(x)

    def training_step(self, batch, batch_idx):
        x = batch
        y = x.y
        y_hat = self(x)
        loss = cross_entropy(y_hat, y)
        acc = accuracy(
            y_hat,
            y,
            task="multiclass",
            num_classes=7
        )

        self.log("train_loss", loss)
        self.log("train_acc", acc)
        return loss

    def validation_step(self, batch, batch_idx):
        x = batch
        y = x.y
        y_hat = self(x)
        loss = cross_entropy(y_hat, y)
        acc = accuracy(
            y_hat,
            y,
            task="multiclass",
            num_classes=7
        )
        self.log("valid_loss", loss)
        self.log("valid_acc", acc)

    def test_step(self, batch, batch_idx):
        x = batch
        y = x.y
        y_hat = self(x)
        loss = cross_entropy(y_hat, y)
        acc = accuracy(
            y_hat,
            y,
            task="multiclass",
            num_classes=7
        )

        y_pred_softmax = torch.log_softmax(y_hat, dim = 1)
        y_pred_tags = torch.argmax(y_pred_softmax, dim = 1)
        f1 = f1_score(y.detach().cpu().numpy(), y_pred_tags.detach().cpu().numpy(), average = 'weighted')

        self.log("test_loss", loss)
        self.log("test_acc", acc)
        self.log("test_f1", f1)

    def configure_optimizers(self):
        optimizer = torch.optim.Adam(self.parameters(), lr=config.lr)
        return optimizer

In [121]:
GraphNets()

GraphNets(
  (layer1): GCNConv(3, 8)
  (layer2): GCNConv(8, 8)
  (decoder): Linear(in_features=8, out_features=7, bias=True)
)

In [122]:
from pytorch_lightning.callbacks import ModelCheckpoint
import os

file_path = './graphein_model'
if not os.path.exists(file_path):
    os.mkdir(file_path)

checkpoint_callback = ModelCheckpoint(
    monitor="valid_loss",
    dirpath=file_path,
    filename="model-{epoch:02d}-{val_loss:.2f}",
    save_top_k=1,
    mode="min",
)

In [123]:
model = GraphNets()
trainer = pl.Trainer(max_epochs=100, accelerator='auto', callbacks=[checkpoint_callback])
trainer.fit(model, train_loader, valid_loader)


GPU available: True (cuda), used: True
TPU available: False, using: 0 TPU cores
💡 Tip: For seamless cloud logging and experiment tracking, try installing [litlogger](https://pypi.org/project/litlogger/) to enable LitLogger, which logs metrics and artifacts automatically to the Lightning Experiments platform.
LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0]


┏━━━┳━━━━━━━━━┳━━━━━━━━━┳━━━━━━━━┳━━━━━━━┳━━━━━━━┓
┃   ┃ Name    ┃ Type    ┃ Params ┃ Mode  ┃ FLOPs ┃
┡━━━╇━━━━━━━━━╇━━━━━━━━━╇━━━━━━━━╇━━━━━━━╇━━━━━━━┩
│ 0 │ layer1  │ GCNConv │     32 │ train │     0 │
│ 1 │ layer2  │ GCNConv │     72 │ train │     0 │
│ 2 │ decoder │ Linear  │     63 │ train │     0 │
└───┴─────────┴─────────┴────────┴───────┴───────┘

Trainable params: 167                                                                                              
Non-trainable params: 0                                                                                            
Total params: 167                                                                                                  
Total estimated model params size (MB): 0.001                                                                      
Modules in train mode: 7                                                                                           
Modules in eval mode: 0                                                                                            
Total FLOPs: 0

Output()

`Trainer.fit` stopped: `max_epochs=100` reached.


In [124]:
best_model = GraphNets.load_from_checkpoint(checkpoint_callback.best_model_path)
out_best_test = trainer.test(best_model, test_loader)[0]

LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0]


Output()

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━┓
┃        Test metric        ┃       DataLoader 0        ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━┩
│         test_acc          │    0.29567307233810425    │
│          test_f1          │    0.1516140103340149     │
│         test_loss         │     1.869748592376709     │
└───────────────────────────┴───────────────────────────┘

In [125]:
with open("./baseline_time.txt", 'a') as f:
    f.write(f'\n{str(time.time() - running_time)}')